# PrecisionMiner: an Extraction Candidate

Run a structured extraction workflow on a short, controlled paper record. The fixed response lets us inspect retrieval, parsing, and evidence matching without a model or API key.


## Set up a small evidence index

Notebook 02 explains indexing; here a helper builds the same local index so we can focus on the extraction workflow.


In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb

WORK_DIR = nb.bootstrap()


In [ ]:
sample_papers = nb.sample_papers()
list(sample_papers)

In [ ]:
embedder = nb.TinyKeywordEmbedder()
embedder.model_name, embedder.dim

In [ ]:
# Indexing is covered in notebooks 01 and 02; here it is one helper call so the
# notebook can stay focused on the extraction workflow.
vdb, retriever = nb.build_index(WORK_DIR / "index", sample_papers, embedder)

len(vdb.get_points()), vdb.get_embedding_model()

## Configure the extraction task

`FindDataSourcesConfig` supplies retrieval questions and the expected JSON schema. We request three chunks from the selected paper.


In [ ]:
from epilens.workflows.precision_miner import FindDataSourcesConfig

strategy_name = "demo-sections"
academic_db = nb.build_academic_db(sample_papers, strategy_name)

config = FindDataSourcesConfig(top_k=3)
config.retrieval_templates[:2], config.top_k

## Supply an illustrative response

`FixedJSONGenerator` returns this same JSON regardless of the retrieved text. It demonstrates the workflow contract, not model performance. The quote below is taken from the Methods section, so we can check that a retrieved chunk contains it. A real `LLMGenerator` would use the prompt and evidence to propose items; `structured_output="schema"` asks a supporting provider to enforce the JSON shape.


In [ ]:
registry_quote = "The analysis used patient records from the National Hospital Registry."
demo_payload = {
    "description": "One possible data source to review.",
    "items": [
        {
            "name": "National Hospital Registry",
            "url": None,
            "explanation": "The Methods section names this registry as the source of patient records.",
            "raw_text": registry_quote,
        }
    ],
}
generator = nb.FixedJSONGenerator(demo_payload, model_id="fixed-extraction-demo")


## Run and inspect the workflow

`run_detailed` returns the parsed candidate plus the retrieved chunks and raw response. Check that the supplied quote occurs in retrieved evidence before reviewing what the item means.


In [ ]:
from epilens.workflows import PrecisionMiner

miner = PrecisionMiner(
    retriever=retriever,
    generator=generator,
    strategy_name=strategy_name,
    config=config,
    academic_db=academic_db,
)

detailed = miner.run_detailed("paper_open_data")
detailed.result.model_dump()


In [ ]:
print("Retrieved chunks")
for chunk in detailed.relevant_chunks:
    print(f"- {chunk.paper_id} | {chunk.section_title} | {chunk.text[:160]}")

print("\nCandidate and matching evidence")
for item in detailed.result.items:
    matches = [chunk for chunk in detailed.relevant_chunks if item.raw_text in chunk.text]
    print(f"- {item.name}: quote found in {len(matches)} retrieved chunk(s)")
    for chunk in matches:
        print(f"  {chunk.paper_id} | {chunk.section_title}")
    assert matches, f"No retrieved chunk contains the quote for {item.name}"

print("\nRaw fixed response")
print(detailed.trace.raw_llm_response)


The text match checks traceability only. This fixed response is an illustrative worklist for human checking, not a validated fact about the paper or an estimate of how data sources are distributed across studies. Review the original paper section before accepting an item.
